# Imports

In [1]:
from __future__ import annotations

import logging
from pathlib import Path


%load_ext autoreload
%autoreload 2

In [2]:
%cd ..

/Users/simon/PycharmProjects/alpaca_eval


/Users/simon/PycharmProjects/alpaca_eval/.venv/lib/python3.9/site-packages/IPython/core/magics/osm.py:417: UserWarning: using dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [3]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from huggingface_hub import hf_hub_download
from scipy.stats import spearmanr
from statsmodels.stats.outliers_influence import variance_inflation_factor

from alpaca_eval import constants, utils
from alpaca_eval.main import make_leaderboard, evaluate_multiple
from alpaca_eval.metrics.glm_winrate import make_dmatrix_for_model, fit_LogisticRegressionCV
from notebooks.notebook_helpers import get_chatbot_arena_lb_mapping
from notebooks.notebook_helpers import load_annotations

/Users/simon/PycharmProjects/alpaca_eval/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/simon/PycharmProjects/alpaca_eval/.venv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Instruction difficulty download

Download instruction difficulties that were calculated before by training GLM with different fixed terms

In [3]:
out = hf_hub_download(repo_id="tatsu-lab/alpaca_eval",
                      filename="instruction_difficulty.csv",
                      repo_type="dataset",
                      force_download=True)

out

'/Users/simon/.cache/huggingface/hub/datasets--tatsu-lab--alpaca_eval/snapshots/2edc6fad8be6b14ea7230aabfd08188da6b8b814/instruction_difficulty.csv'

In [4]:
hf_instruction_difficulty = pd.read_csv(out, index_col=0).squeeze()
hf_instruction_difficulty

index
0      0.000000
1     -0.101360
2     -1.356212
3      0.700087
4     -1.597457
         ...   
800   -0.272119
801    0.437628
802    1.841663
803   -0.010396
804   -0.656699
Name: instruction_difficulty, Length: 805, dtype: float64

# Extract results of evaluating (annotations)

In [5]:
lb = pd.read_csv("src/alpaca_eval/leaderboards/data_AlpacaEval_2/weighted_alpaca_eval_gpt4_turbo_leaderboard.csv",
                 index_col=0)

lb

,win_rate,standard_error,n_wins,n_wins_base,n_draws,n_total,discrete_win_rate,mode,avg_length,length_controlled_winrate,lc_standard_error
NullModel,76.919792,0.909010,676,129,0,805,83.975155,community,872,86.457807,0.141800
SelfMoA_gemma-2-9b-it-WPO-HB,77.589552,1.231941,640,165,0,805,79.503106,community,3261,78.539281,0.304279
Shopee-SlimMoA-v1,75.614287,1.270627,621,184,0,805,77.142857,community,1994,77.451543,0.430175
blendaxai-gm-l6-vo31,69.110335,1.328074,562,242,1,805,69.875776,community,1809,76.919812,0.572537
gemma-2-9b-it-WPO-HB,77.825032,1.235586,640,163,2,805,79.627329,community,2285,76.725068,0.424260
...,...,...,...,...,...,...,...,...,...,...,...
oasst-sft-pythia-12b,1.790114,0.398558,13,790,2,805,1.739130,verified,726,3.270102,NaN
guanaco-13b,3.469597,0.551861,22,780,3,805,2.919255,verified,1774,3.003787,NaN
guanaco-7b,2.880002,0.520292,21,783,1,805,2.670807,verified,1364,2.871117,NaN
Qwen1.5-1.8B-Chat,3.705557,0.581175,27,774,3,804,3.544776,verified,2673,2.588499,NaN


For every assessed model, extract results of **805** questions answers vs baseline answers

In [7]:
all_df_annotations = load_annotations(lb)
all_df_annotations = all_df_annotations.query("len_2 != 0").dropna(axis=1)

all_df_annotations

,index,generator_1,generator_2,annotator,preference,len_1,len_2,is_longer2,is_longer1,is_same_length,model,position_component
0,0,gpt4_1106_preview,NullModel,weighted_alpaca_eval_gpt4_turbo,0.990014,2104,872,False,True,False,NullModel,1
1,1,gpt4_1106_preview,NullModel,weighted_alpaca_eval_gpt4_turbo,0.459366,3394,872,False,True,False,NullModel,1
2,2,gpt4_1106_preview,NullModel,weighted_alpaca_eval_gpt4_turbo,0.066485,3025,872,False,True,False,NullModel,1
3,3,gpt4_1106_preview,NullModel,weighted_alpaca_eval_gpt4_turbo,0.881640,2908,872,False,True,False,NullModel,-1
4,4,gpt4_1106_preview,NullModel,weighted_alpaca_eval_gpt4_turbo,0.948271,2341,872,False,True,False,NullModel,-1
...,...,...,...,...,...,...,...,...,...,...,...,...
169032,800,gpt4_1106_preview,baichuan-13b-chat,weighted_alpaca_eval_gpt4_turbo,0.000005,5164,2908,False,True,False,baichuan-13b-chat,1
169033,801,gpt4_1106_preview,baichuan-13b-chat,weighted_alpaca_eval_gpt4_turbo,0.000001,4416,4869,True,False,False,baichuan-13b-chat,1
169034,802,gpt4_1106_preview,baichuan-13b-chat,weighted_alpaca_eval_gpt4_turbo,0.000064,3070,1308,False,True,False,baichuan-13b-chat,-1
169035,803,gpt4_1106_preview,baichuan-13b-chat,weighted_alpaca_eval_gpt4_turbo,0.000003,3769,2341,False,True,False,baichuan-13b-chat,1


Add instruction difficulties

In [8]:
all_df_annotations['instruction_difficulty'] = all_df_annotations['index'].transform(
    lambda i: hf_instruction_difficulty[i])
all_df_annotations

,index,generator_1,generator_2,annotator,preference,len_1,len_2,is_longer2,is_longer1,is_same_length,model,position_component,instruction_difficulty
0,0,gpt4_1106_preview,NullModel,weighted_alpaca_eval_gpt4_turbo,0.990014,2104,872,False,True,False,NullModel,1,0.000000
1,1,gpt4_1106_preview,NullModel,weighted_alpaca_eval_gpt4_turbo,0.459366,3394,872,False,True,False,NullModel,1,-0.101360
2,2,gpt4_1106_preview,NullModel,weighted_alpaca_eval_gpt4_turbo,0.066485,3025,872,False,True,False,NullModel,1,-1.356212
3,3,gpt4_1106_preview,NullModel,weighted_alpaca_eval_gpt4_turbo,0.881640,2908,872,False,True,False,NullModel,-1,0.700087
4,4,gpt4_1106_preview,NullModel,weighted_alpaca_eval_gpt4_turbo,0.948271,2341,872,False,True,False,NullModel,-1,-1.597457
...,...,...,...,...,...,...,...,...,...,...,...,...,...
169032,800,gpt4_1106_preview,baichuan-13b-chat,weighted_alpaca_eval_gpt4_turbo,0.000005,5164,2908,False,True,False,baichuan-13b-chat,1,-0.272119
169033,801,gpt4_1106_preview,baichuan-13b-chat,weighted_alpaca_eval_gpt4_turbo,0.000001,4416,4869,True,False,False,baichuan-13b-chat,1,0.437628
169034,802,gpt4_1106_preview,baichuan-13b-chat,weighted_alpaca_eval_gpt4_turbo,0.000064,3070,1308,False,True,False,baichuan-13b-chat,-1,1.841663
169035,803,gpt4_1106_preview,baichuan-13b-chat,weighted_alpaca_eval_gpt4_turbo,0.000003,3769,2341,False,True,False,baichuan-13b-chat,1,-0.010396


Extract the name of baseline

In [9]:
BASELINE_NAME = all_df_annotations['generator_1'].unique()[0]
BASELINE_NAME

'gpt4_1106_preview'

Drop useless features

In [10]:
all_df_annotations_dropped = all_df_annotations.drop(
    ['model', 'generator_1', 'is_longer1', 'is_longer2', 'is_same_length', 'annotator', 'index'], axis=1)
all_df_annotations_dropped

,generator_2,preference,len_1,len_2,position_component,instruction_difficulty
0,NullModel,0.990014,2104,872,1,0.000000
1,NullModel,0.459366,3394,872,1,-0.101360
2,NullModel,0.066485,3025,872,1,-1.356212
3,NullModel,0.881640,2908,872,-1,0.700087
4,NullModel,0.948271,2341,872,-1,-1.597457
...,...,...,...,...,...,...
169032,baichuan-13b-chat,0.000005,5164,2908,1,-0.272119
169033,baichuan-13b-chat,0.000001,4416,4869,1,0.437628
169034,baichuan-13b-chat,0.000064,3070,1308,-1,1.841663
169035,baichuan-13b-chat,0.000003,3769,2341,1,-0.010396


# Train dataset preparation

Calculate len diff component by formula `diff_component = np.tanh((len_2 - len_1) / std((len_2 - len_1)))` + drop useless components 

In [11]:
train_df = all_df_annotations_dropped.copy()

train_df['len_diff'] = train_df['len_2'] - train_df['len_1']
std_diffs = train_df['len_diff'].std()

train_df['diff_component'] = np.tanh(train_df['len_diff'] / std_diffs)

train_df = train_df.drop(['len_1', 'len_2', 'len_diff'], axis=1)
train_df

,generator_2,preference,position_component,instruction_difficulty,diff_component
0,NullModel,0.990014,1,0.000000,-0.810112
1,NullModel,0.459366,1,-0.101360,-0.980401
2,NullModel,0.066485,1,-1.356212,-0.961855
3,NullModel,0.881640,-1,0.700087,-0.952961
4,NullModel,0.948271,-1,-1.597457,-0.872683
...,...,...,...,...,...
169032,baichuan-13b-chat,0.000005,1,-0.272119,-0.968305
169033,baichuan-13b-chat,0.000001,1,0.437628,0.392306
169034,baichuan-13b-chat,0.000064,-1,1.841663,-0.923505
169035,baichuan-13b-chat,0.000003,1,-0.010396,-0.863440


Prepare test dataset (temporary, **not used right now**)

In [13]:
test_df = train_df.copy()
test_df['diff_component'] = 0

test_df

,generator_2,preference,position_component,instruction_difficulty,diff_component
0,NullModel,0.990014,1,0.000000,0
1,NullModel,0.459366,1,-0.101360,0
2,NullModel,0.066485,1,-1.356212,0
3,NullModel,0.881640,-1,0.700087,0
4,NullModel,0.948271,-1,-1.597457,0
...,...,...,...,...,...
169032,baichuan-13b-chat,0.000005,1,-0.272119,0
169033,baichuan-13b-chat,0.000001,1,0.437628,0
169034,baichuan-13b-chat,0.000064,-1,1.841663,0
169035,baichuan-13b-chat,0.000003,1,-0.010396,0


Extract `patsy` matrices for the logistic regression train

In [14]:
formula = f"C(generator_2, Treatment(reference='{BASELINE_NAME}')) + diff_component + instruction_difficulty + position_component - 1"

df_XY_train, _ = make_dmatrix_for_model(train_df, test_df, formula)
df_XY_train

,"C(generator_2, Treatment(reference='gpt4_1106_preview'))[Conifer-7B-DPO]","C(generator_2, Treatment(reference='gpt4_1106_preview'))[Contextual-KTO-Mistral-PairRM]","C(generator_2, Treatment(reference='gpt4_1106_preview'))[FsfairX-Zephyr-Chat-v0.1]","C(generator_2, Treatment(reference='gpt4_1106_preview'))[Infinity-Instruct-3M-0613-Llama3-70B]","C(generator_2, Treatment(reference='gpt4_1106_preview'))[Infinity-Instruct-3M-0613-Mistral-7B]","C(generator_2, Treatment(reference='gpt4_1106_preview'))[Infinity-Instruct-3M-0625-Llama3-70B]","C(generator_2, Treatment(reference='gpt4_1106_preview'))[Infinity-Instruct-3M-0625-Llama3-8B]","C(generator_2, Treatment(reference='gpt4_1106_preview'))[Infinity-Instruct-3M-0625-Mistral-7B]","C(generator_2, Treatment(reference='gpt4_1106_preview'))[Infinity-Instruct-3M-0625-Qwen2-7B]","C(generator_2, Treatment(reference='gpt4_1106_preview'))[Infinity-Instruct-3M-0625-Yi-1.5-9B]",...,"C(generator_2, Treatment(reference='gpt4_1106_preview'))[xwinlm-7b-v0.1]","C(generator_2, Treatment(reference='gpt4_1106_preview'))[yi-large-preview]","C(generator_2, Treatment(reference='gpt4_1106_preview'))[zephyr-7b-alpha]","C(generator_2, Treatment(reference='gpt4_1106_preview'))[zephyr-7b-alpha-ExPO]","C(generator_2, Treatment(reference='gpt4_1106_preview'))[zephyr-7b-beta]","C(generator_2, Treatment(reference='gpt4_1106_preview'))[zephyr-7b-beta-ExPO]",diff_component,instruction_difficulty,position_component,preference
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,-0.810112,0.000000,1.0,0.990014
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,-0.980401,-0.101360,1.0,0.459366
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,-0.961855,-1.356212,1.0,0.066485
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,-0.952961,0.700087,-1.0,0.881640
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,-0.872683,-1.597457,-1.0,0.948271
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
169032,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,-0.968305,-0.272119,1.0,0.000005
169033,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.392306,0.437628,1.0,0.000001
169034,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,-0.923505,1.841663,-1.0,0.000064
169035,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,-0.863440,-0.010396,1.0,0.000003


Train logistic regression with the prepared matrix (it's just a model, not a statistical analysis - based on `sklearn`). 
Training features:
- Intercept term is disabled (fit_intercept=False) because the categorical generator variable already encodes model-level offsets (θₘ), and adding a separate intercept would introduce redundancy.
- Logistic regression with **L2 regularization** is used to prevent overfitting and penalize extreme coefficient values, particularly for the length bias term.
- **Cross-entropy** (log loss) is the optimization objective, ensuring the model learns probabilistic preferences accurately.
- Probabilistic labels (preference ∈ [0, 1]) are handled by duplicating each data point: once with label 1 and weight = p, once with label 0 and weight = 1 - p. This allows the model to optimize log loss directly on soft targets.
- Cross-validation (5-fold) is used during training to ensure the robustness of learned coefficients and regularization.

In [15]:
model = fit_LogisticRegressionCV(
    data=df_XY_train,
    col_y_true="preference",
    is_ytrue_proba=True,
    n_splits=5,
    penalty="l2",
    solver="lbfgs",
    fit_intercept=False
)

model

LogisticRegressionCV(cv=GroupKFold(n_splits=5, random_state=None, shuffle=False),
                     fit_intercept=False, random_state=123,
                     scoring=make_scorer(logloss_continuous, greater_is_better=False, response_method='predict_proba'))

Final coefficient for **position bias term**

In [16]:
coef_df = pd.DataFrame({
    "feature": model.feature_names_in_,
    "coef": model.coef_.flatten()
})

coef_df[coef_df['feature'] == 'position_component']

,feature,coef
212,position_component,-0.074063


# Significance check
Since `scikit‑learn’s LogisticRegression` does not support statistical inference on model coefficients, I switched to using `statsmodels` to fit a logistic (logit) regression and obtain **p‑values** for each predictor, omitting both cross‑validation and regularization in that step.

The threshold you choose for deciding whether a p‑value is “significant” is called the significance level

In [17]:
SIGNIFICANCE_LEVEL = 0.05

Change to binominal preference

In [18]:
binominal_train_df = train_df.copy()
binominal_train_df['preference'] = (binominal_train_df['preference'] >= 0.5).astype(int)

binominal_train_df

,generator_2,preference,position_component,instruction_difficulty,diff_component
0,NullModel,1,1,0.000000,-0.810112
1,NullModel,0,1,-0.101360,-0.980401
2,NullModel,0,1,-1.356212,-0.961855
3,NullModel,1,-1,0.700087,-0.952961
4,NullModel,1,-1,-1.597457,-0.872683
...,...,...,...,...,...
169032,baichuan-13b-chat,0,1,-0.272119,-0.968305
169033,baichuan-13b-chat,0,1,0.437628,0.392306
169034,baichuan-13b-chat,0,-1,1.841663,-0.923505
169035,baichuan-13b-chat,0,1,-0.010396,-0.863440


In [19]:
binominal_train_df.describe()

,preference,position_component,instruction_difficulty,diff_component
count,169008.000000,169008.000000,169008.000000,169008.000000
mean,0.209245,0.011029,-0.057163,-0.316035
std,0.406770,0.999942,1.480527,0.513870
min,0.000000,-1.000000,-3.052593,-0.999989
25%,0.000000,-1.000000,-1.152511,-0.776607
50%,0.000000,1.000000,-0.229853,-0.380631
75%,0.000000,1.000000,0.786168,0.018299
max,1.000000,1.000000,5.811319,1.000000


In [20]:
binominal_train_df.dtypes

generator_2                object
preference                  int64
position_component          int64
instruction_difficulty    float64
diff_component            float64
dtype: object

## Assumptions check

Before we can apply logistic regression as a statistical tool (with only random slope in the case of AlpacaEval), we should test its assumptions.

These include: **the binary nature of the dependent variable**, **linearity between predictors and the log-odds**, **absence of multicollinearity**, **independence of observations**, and **sufficient sample size**.

In our case sufficient sample size and independence of observations are already satisfied by AlpacaEval environment

Therefore, in our setting, we primarily focus on two critical assumptions:
 - **(1)** the absence of multicollinearity between predictors, as correlated features can destabilize coefficient estimates, and  
- **(2)** the linearity between continuous predictors and the log-odds of the outcome, as violating this assumption can lead to biased model estimates and misinterpretation of predictor effects.

We will drop the nominal feature since it will be one-hot encoded and can't correlate with the rest of feature from the logical point of view

In [21]:
X = binominal_train_df.copy()
X = X.drop(columns=['generator_2'])

X.describe()

,preference,position_component,instruction_difficulty,diff_component
count,169008.000000,169008.000000,169008.000000,169008.000000
mean,0.209245,0.011029,-0.057163,-0.316035
std,0.406770,0.999942,1.480527,0.513870
min,0.000000,-1.000000,-3.052593,-0.999989
25%,0.000000,-1.000000,-1.152511,-0.776607
50%,0.000000,1.000000,-0.229853,-0.380631
75%,0.000000,1.000000,0.786168,0.018299
max,1.000000,1.000000,5.811319,1.000000


Check for multi-collinearity (using VIF as an identification)

In [22]:
vif_data = pd.DataFrame()
vif_data["feature"] = X.columns
vif_data["VIF"] = [variance_inflation_factor(X.values, i)
                   for i in range(X.shape[1])]

vif_data

,feature,VIF
0,preference,1.135976
1,position_component,1.207763
2,instruction_difficulty,1.363798
3,diff_component,1.022866


As we can see, the VIF factor is no more than **5** => there is no multi-collinearity 

## Model training

Train the model with the same `patsy` formula (including target variable):
- only random slope strategy is used to duplicate the method of training from LC AlpacaEval

In [23]:
result = smf.logit(
    formula=f"preference ~ {formula}",
    data=binominal_train_df,
).fit()

result

         Current function value: 0.273098
         Iterations: 35


/Users/simon/PycharmProjects/alpaca_eval/.venv/lib/python3.9/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


We are testing the position bias component (`position_component`) and see that its p-value is below our chosen threshold, indicating that the position bias effect is statistically significant.

In [24]:
summary_df = result.summary2().tables[1]
diff_row = summary_df.loc["position_component"]

diff_row

Coef.      -7.487726e-02
Std.Err.    9.065905e-03
z          -8.259215e+00
P>|z|       1.466332e-16
[0.025     -9.264610e-02
0.975]     -5.710841e-02
Name: position_component, dtype: float64

In [25]:
bool(diff_row['P>|z|'] < SIGNIFICANCE_LEVEL)

True

# Gamed features (df_gamed.csv) 

In [26]:
final_gamed_df = pd.DataFrame()

In [27]:
GAMED_MODELS = ['gpt4_1106_preview_verbose', 'gpt4_1106_preview_concise']

In [28]:
out_dir = utils.get_output_path(
    "auto",
    "results/gpt4_gamed/model_outputs.json",
    "gpt4_gamed",
    annotators_config=constants.DEFAULT_ANNOTATOR_CONFIG,  # must match original
) / "annotations.json"

out_dir

PosixPath('results/gpt4_gamed/weighted_alpaca_eval_gpt4_turbo/annotations.json')

In [30]:
for model in GAMED_MODELS:
    gamed_out_df = pd.read_json(f'results/{model}/weighted_alpaca_eval_gpt4_turbo/annotations.json')
    df = gamed_out_df.reset_index()
    len_1 = df["output_1"].str.len()
    len_2 = df["output_2"].str.len()
    std_delta_len = len_1 - len_2
    df = df[["preference", "index"]].copy()
    df["std_delta_len"] = std_delta_len / std_delta_len.std()

    if "position_component" in gamed_out_df.columns:
        df["position_component"] = gamed_out_df["position_component"].values
    elif "raw_completion" in gamed_out_df.columns:
        def first_token(raw):
            try:
                return raw["logprobs"]["content"][0]["token"]
            except Exception:
                return None


        tokens = gamed_out_df["raw_completion"].apply(first_token)
        pref_raw = gamed_out_df["preference"].astype(float)
        mask_lower = tokens == "m"
        df["position_component"] = np.where(
            mask_lower,
            np.where(pref_raw >= 1.5, 1, 0),
            np.where(pref_raw < 1.5, 1, 0),
        )
    else:
        df["position_component"] = 0.0

    df['position_length_interaction'] = df["position_component"] * df["std_delta_len"]
    df['preference'] = df['preference'] - 1
    df['not_gamed_baseline'] = False

    final_gamed_df = pd.concat([final_gamed_df, df], ignore_index=True)

final_gamed_df

,preference,index,std_delta_len,position_component,position_length_interaction,not_gamed_baseline
0,0.997285,0,-1.069349,1,-1.069349,False
1,0.984094,1,-7.604716,1,-7.604716,False
2,0.085099,2,0.234434,1,0.234434,False
3,0.990587,3,-1.192735,0,-0.000000,False
4,0.839734,4,-0.686851,0,-0.000000,False
...,...,...,...,...,...,...
1605,0.300746,800,2.845325,1,2.845325,False
1606,0.658418,801,1.873407,1,1.873407,False
1607,0.946597,802,1.493091,0,0.000000,False
1608,0.004070,803,0.948442,1,0.948442,False


In [31]:
final_gamed_df.to_csv('./df_gamed.csv', index=False)

# Additional library

In [4]:
# Sync the models based on existence of annotation and equality between generator and name of the model
def get_all_res(src_lb: str):
    models = pd.read_csv(src_lb, index_col=0).index.to_list()
    model_outputs_list: list[str] = []

    for name in models:
        model_outputs_path = Path(f"results/{name}/model_outputs.json")
        annotations_path = model_outputs_path.parent / "weighted_alpaca_eval_gpt4_turbo" / "annotations.json"

        if not annotations_path.exists():
            logging.warning(f"Skip {name}: missing {annotations_path}")
            continue

        try:
            gen = pd.read_json(model_outputs_path)["generator"].iloc[0]
        except Exception:
            logging.warning(msg=f"Skip {name}: unreadable {model_outputs_path}")
            continue

        if gen != name:
            logging.warning(msg=f"Skip {name}: generator mismatch ({gen} != {name})")
            continue

        model_outputs_list.append(str(model_outputs_path))

    logging.info(f"Gathered {len(model_outputs_list)} of {len(models)} models")

    return model_outputs_list

In [5]:
def evaluate_model_list(
        src_lb: str,
        lb_path: str,
        metric_kwargs=None,
        fn_metric: str = "get_length_controlled_winrate",
):
    """
    Recompute the leaderboard using make_leaderboard and a lazy all_model_outputs provider.

    - Gathers only those results/<model>/model_outputs.json for which
      results/<model>/weighted_alpaca_eval_gpt4_turbo/annotations.json exists
      AND the JSON 'generator' matches the model name.

    - Calls make_leaderboard with is_recompute_metrics_only=True and saves to lb_path.
    """
    metric_kwargs = metric_kwargs or {}

    outputs = get_all_res(src_lb)

    # Run leaderboard recomputation
    df, _ = make_leaderboard(
        leaderboard_path=lb_path,
        all_model_outputs=outputs,  # pass the callable, not its result
        is_recompute_metrics_only=True,
        fn_metric=fn_metric,
        sort_by="length_controlled_winrate",
        output_path="auto",
        is_return_instead_of_print=True,
        is_cache_leaderboard=False,
        metric_kwargs=metric_kwargs,
    )

    # Ensure saved and print a short summary (make_leaderboard already writes to lb_path)
    if isinstance(df, pd.DataFrame):
        print(f"Saved recomputed leaderboard → {lb_path}\nRecomputed {len(df)} models.")

    return df

In [6]:
def calculate_spearman_correlation(lb_path):
    lb = pd.read_csv(lb_path, index_col=0)
    
    arena = pd.Series(get_chatbot_arena_lb_mapping())  # {model: arena_elo}
    
    common = lb.index.intersection(arena.index)
    lc = lb.loc[common, "length_controlled_winrate"]
    elo = arena.loc[common]
    
    s = spearmanr(lc, elo).statistic
    print(f"Spearman (LC AlpacaEval vs Chatbot Arena): {s:.4f}")

In [7]:
def merge_all_datasets(annotation_datasets):
    dfs = [pd.read_json(dataset) for dataset in annotation_datasets]
    return pd.concat(dfs, ignore_index=True)

# Reranking the rating (LC AlpacaEval)

Define initial leaderboard that will be (partially) recalculated

In [8]:
LC_SRC_LB = "src/alpaca_eval/leaderboards/data_AlpacaEval_2/weighted_alpaca_eval_gpt4_turbo_leaderboard.csv"

Rerank all the models with LC AlpacaEval strategy

In [63]:
lc_lb_path = "src/alpaca_eval/leaderboards/recomputed_weighted_alpaca_eval_gpt4_turbo.csv"

evaluate_model_list(
    LC_SRC_LB,
    lc_lb_path,
    metric_kwargs={"glm_name": "length_controlled_v1"}
)

INFO:root:Gathered 213 of 223 models
INFO:root:Evaluating the SelfMoA_gemma-2-9b-it-WPO-HB outputs.
INFO:root:Saving all results to results/SelfMoA_gemma-2-9b-it-WPO-HB/weighted_alpaca_eval_gpt4_turbo/weighted_alpaca_eval_gpt4_turbo


     np.tanh(std_delta_len)  instruction_difficulty  \
0                 -0.789216                0.000000   
1                 -1.000000               -0.101360   
2                  0.230232               -1.356212   
3                 -0.831425                0.700087   
4                 -0.595955               -1.597457   
..                      ...                     ...   
800                0.068190               -0.272119   
801                0.219576                0.437628   
802               -0.084749                1.841663   
803                0.052423               -0.010396   
804                0.345039               -0.656699   

     not_gamed_baseline.astype(float)  preference  
0                                 0.0    0.997285  
1                                 0.0    0.984094  
2                                 0.0    0.085099  
3                                 0.0    0.990587  
4                                 0.0    0.839734  
..                         

INFO:root:Evaluating the Shopee-SlimMoA-v1 outputs.
INFO:root:Saving all results to results/Shopee-SlimMoA-v1/weighted_alpaca_eval_gpt4_turbo/weighted_alpaca_eval_gpt4_turbo


     np.tanh(std_delta_len)  instruction_difficulty  \
0                 -0.789216                0.000000   
1                 -1.000000               -0.101360   
2                  0.230232               -1.356212   
3                 -0.831425                0.700087   
4                 -0.595955               -1.597457   
..                      ...                     ...   
800               -0.279256               -0.272119   
801                0.951701                0.437628   
802                0.630361                1.841663   
803               -0.875406               -0.010396   
804                0.976707               -0.656699   

     not_gamed_baseline.astype(float)  preference  
0                                 0.0    0.997285  
1                                 0.0    0.984094  
2                                 0.0    0.085099  
3                                 0.0    0.990587  
4                                 0.0    0.839734  
..                         

INFO:root:Evaluating the blendaxai-gm-l6-vo31 outputs.
INFO:root:Saving all results to results/blendaxai-gm-l6-vo31/weighted_alpaca_eval_gpt4_turbo/weighted_alpaca_eval_gpt4_turbo


...

INFO:root:Evaluating the SelfMoA_gemma-2-9b-it-SimPO outputs.
INFO:root:Saving all results to results/SelfMoA_gemma-2-9b-it-SimPO/weighted_alpaca_eval_gpt4_turbo/weighted_alpaca_eval_gpt4_turbo


     np.tanh(std_delta_len)  instruction_difficulty  \
0                 -0.789216                0.000000   
1                 -1.000000               -0.101360   
2                  0.230232               -1.356212   
3                 -0.831425                0.700087   
4                 -0.595955               -1.597457   
..                      ...                     ...   
800                0.666048               -0.272119   
801                0.853891                0.437628   
802                0.766566                1.841663   
803                0.296565               -0.010396   
804                0.946313               -0.656699   

     not_gamed_baseline.astype(float)  preference  
0                                 0.0    0.997285  
1                                 0.0    0.984094  
2                                 0.0    0.085099  
3                                 0.0    0.990587  
4                                 0.0    0.839734  
..                         

INFO:root:Evaluating the blendaxai-gm-l3-v35 outputs.
INFO:root:Saving all results to results/blendaxai-gm-l3-v35/weighted_alpaca_eval_gpt4_turbo/weighted_alpaca_eval_gpt4_turbo


     np.tanh(std_delta_len)  instruction_difficulty  \
0                 -0.789216                0.000000   
1                 -1.000000               -0.101360   
2                  0.230232               -1.356212   
3                 -0.831425                0.700087   
4                 -0.595955               -1.597457   
..                      ...                     ...   
800                0.917810               -0.272119   
801                0.775299                0.437628   
802                0.550997                1.841663   
803               -0.713019               -0.010396   
804                0.977038               -0.656699   

     not_gamed_baseline.astype(float)  preference  
0                                 0.0    0.997285  
1                                 0.0    0.984094  
2                                 0.0    0.085099  
3                                 0.0    0.990587  
4                                 0.0    0.839734  
..                         

KeyboardInterrupt: 

Calculate the spearman correlation

In [42]:
calculate_spearman_correlation(lc_lb_path)

Spearman (LC AlpacaEval vs Chatbot Arena): 0.9746


# Reranking the rating (LPC AlpacaEval)

Rerank all the models with LPC AlpacaEval strategy (with [0;1] encoding)

In [43]:
lb_path_pos = "src/alpaca_eval/leaderboards/recomputed_weighted_alpaca_eval_gpt4_turbo_positioned.csv"

evaluate_model_list(LC_SRC_LB,
                    lb_path_pos,
                    metric_kwargs={"glm_name": "length_position_controlled_v1"}
                    )

INFO:root:Gathered 213 of 223 models
INFO:root:Evaluating the SelfMoA_gemma-2-9b-it-WPO-HB outputs.
INFO:root:Saving all results to results/SelfMoA_gemma-2-9b-it-WPO-HB/weighted_alpaca_eval_gpt4_turbo/weighted_alpaca_eval_gpt4_turbo
INFO:root:Evaluating the Shopee-SlimMoA-v1 outputs.
INFO:root:Saving all results to results/Shopee-SlimMoA-v1/weighted_alpaca_eval_gpt4_turbo/weighted_alpaca_eval_gpt4_turbo
INFO:root:Evaluating the blendaxai-gm-l6-vo31 outputs.
INFO:root:Saving all results to results/blendaxai-gm-l6-vo31/weighted_alpaca_eval_gpt4_turbo/weighted_alpaca_eval_gpt4_turbo
INFO:root:Evaluating the gemma-2-9b-it-WPO-HB outputs.
INFO:root:Saving all results to results/gemma-2-9b-it-WPO-HB/weighted_alpaca_eval_gpt4_turbo/weighted_alpaca_eval_gpt4_turbo
INFO:root:Evaluating the SelfMoA_gemma-2-9b-it-SimPO outputs.
INFO:root:Saving all results to results/SelfMoA_gemma-2-9b-it-SimPO/weighted_alpaca_eval_gpt4_turbo/weighted_alpaca_eval_gpt4_turbo
INFO:root:Evaluating the blendaxai-gm-l

Saved recomputed leaderboard → src/alpaca_eval/leaderboards/recomputed_weighted_alpaca_eval_gpt4_turbo_positioned.csv
Recomputed 218 models.


,win_rate,standard_error,n_wins,n_wins_base,n_draws,n_total,discrete_win_rate,mode,avg_length,length_controlled_winrate,lc_standard_error
NullModel,76.919792,0.909010,676,129,0,805,83.975155,community,872,90.170415,0.072275
Shopee-SlimMoA-v1,75.614287,1.270627,621,184,0,805,77.142857,community,1994,80.878184,0.346592
blendaxai-gm-l6-vo31,69.110335,1.328074,562,242,1,805,69.875776,community,1809,80.460486,0.479804
gemma-2-9b-it-WPO-HB,77.825032,1.235586,640,163,2,805,79.627329,community,2285,79.467155,0.360085
openpipe-moa-gpt-4-turbo-v1,63.154935,1.422980,515,283,7,805,64.409938,community,1856,79.412172,0.494603
...,...,...,...,...,...,...,...,...,...,...,...
oasst-sft-pythia-12b,1.790114,0.398558,13,790,2,805,1.739130,verified,726,3.671296,0.207577
guanaco-13b,3.469597,0.551861,22,780,3,805,2.919255,verified,1774,3.155981,0.206464
guanaco-7b,2.880002,0.520292,21,783,1,805,2.670807,verified,1364,3.076146,0.204192
Qwen1.5-1.8B-Chat,3.705557,0.581175,27,774,3,804,3.544776,verified,2673,2.785028,0.201994


Calculate Spearman correlation

In [44]:
calculate_spearman_correlation(lb_path_pos)

Spearman (LC AlpacaEval vs Chatbot Arena): 0.9752


# Reranking the rating (LPC + interaction term AlpacaEval)

Rerank all the models with LPC AlpacaEval strategy + interaction term (with [0;1] encoding)

In [45]:
lb_path_pos_interaction = "src/alpaca_eval/leaderboards/recomputed_weighted_alpaca_eval_gpt4_turbo_positioned_interaction.csv"

evaluate_model_list(LC_SRC_LB, lb_path_pos_interaction,
                    metric_kwargs={"glm_name": "length_position_controlled_v2_interaction"})

INFO:root:Gathered 213 of 223 models
INFO:root:Evaluating the SelfMoA_gemma-2-9b-it-WPO-HB outputs.
INFO:root:Saving all results to results/SelfMoA_gemma-2-9b-it-WPO-HB/weighted_alpaca_eval_gpt4_turbo/weighted_alpaca_eval_gpt4_turbo
INFO:root:Evaluating the Shopee-SlimMoA-v1 outputs.
INFO:root:Saving all results to results/Shopee-SlimMoA-v1/weighted_alpaca_eval_gpt4_turbo/weighted_alpaca_eval_gpt4_turbo
INFO:root:Evaluating the blendaxai-gm-l6-vo31 outputs.
INFO:root:Saving all results to results/blendaxai-gm-l6-vo31/weighted_alpaca_eval_gpt4_turbo/weighted_alpaca_eval_gpt4_turbo
INFO:root:Evaluating the gemma-2-9b-it-WPO-HB outputs.
INFO:root:Saving all results to results/gemma-2-9b-it-WPO-HB/weighted_alpaca_eval_gpt4_turbo/weighted_alpaca_eval_gpt4_turbo
INFO:root:Evaluating the SelfMoA_gemma-2-9b-it-SimPO outputs.
INFO:root:Saving all results to results/SelfMoA_gemma-2-9b-it-SimPO/weighted_alpaca_eval_gpt4_turbo/weighted_alpaca_eval_gpt4_turbo
INFO:root:Evaluating the blendaxai-gm-l

Saved recomputed leaderboard → src/alpaca_eval/leaderboards/recomputed_weighted_alpaca_eval_gpt4_turbo_positioned_interaction.csv
Recomputed 218 models.


,win_rate,standard_error,n_wins,n_wins_base,n_draws,n_total,discrete_win_rate,mode,avg_length,length_controlled_winrate,lc_standard_error
NullModel,76.919792,0.909010,676,129,0,805,83.975155,community,872,90.681264,0.068043
Shopee-SlimMoA-v1,75.614287,1.270627,621,184,0,805,77.142857,community,1994,80.888882,0.346382
blendaxai-gm-l6-vo31,69.110335,1.328074,562,242,1,805,69.875776,community,1809,80.677324,0.475331
SelfMoA_gemma-2-9b-it-WPO-HB,77.589552,1.231941,640,165,0,805,79.503106,community,3261,80.503369,0.282577
openpipe-moa-gpt-4-turbo-v1,63.154935,1.422980,515,283,7,805,64.409938,community,1856,79.265823,0.495872
...,...,...,...,...,...,...,...,...,...,...,...
oasst-sft-pythia-12b,1.790114,0.398558,13,790,2,805,1.739130,verified,726,3.738183,0.210907
guanaco-13b,3.469597,0.551861,22,780,3,805,2.919255,verified,1774,3.233388,0.213187
guanaco-7b,2.880002,0.520292,21,783,1,805,2.670807,verified,1364,3.083273,0.208099
Qwen1.5-1.8B-Chat,3.705557,0.581175,27,774,3,804,3.544776,verified,2673,2.777726,0.205751


Calculate the Spearman correlation 

In [46]:
calculate_spearman_correlation(lb_path_pos_interaction)

Spearman (LC AlpacaEval vs Chatbot Arena): 0.9739


# Final results

In the end (lambda is 0.2):
- **LC AlpacaEval** - 0.9746

position_component - [-1;1]:
- **LPC AlpacaEval** - 0.973
- **LPC AlpacaEval + interaction term** - 0.974

position_component - [0;1]:
- **LPC AlpacaEval** - 0.9752
- **LPC AlpacaEval + interaction term** - 0.9739

Amount of models: 218 of 233 (5 model results are absent)

# Sensitivity analysis 

In [47]:
GLM_INFO_LAMBDAS_SENSITIVITY = [
    0.195,
    0.205,
    0.21,
    0.215
]

In [48]:
BASE_SENSITIVITY_URL = "src/alpaca_eval/leaderboards/data_AlpacaEval_2/sensitivity_analysis"

In [49]:
def process_sensitivity_analysis(formula):
    for idx, lam in enumerate(GLM_INFO_LAMBDAS_SENSITIVITY):
        current_lb_src = f"{BASE_SENSITIVITY_URL}/position_{idx}.csv"

        evaluate_model_list(LC_SRC_LB, current_lb_src, metric_kwargs={"glm_info": {
            "formula": formula,
            "regularize_to_baseline_lambda": lam,
            "kwargs": {"n_splits": 5},
        }})

        print(f'SPEARMAN CORRELATION WITH LAMBDA: {lam}')
        calculate_spearman_correlation(current_lb_src)

## Position component

In [50]:
process_sensitivity_analysis(
    "np.tanh(std_delta_len) + instruction_difficulty + position_component + not_gamed_baseline.astype(float) - 1"
)

INFO:root:Gathered 213 of 223 models
INFO:root:Evaluating the SelfMoA_gemma-2-9b-it-WPO-HB outputs.
INFO:root:Saving all results to results/SelfMoA_gemma-2-9b-it-WPO-HB/weighted_alpaca_eval_gpt4_turbo/weighted_alpaca_eval_gpt4_turbo
INFO:root:Evaluating the Shopee-SlimMoA-v1 outputs.
INFO:root:Saving all results to results/Shopee-SlimMoA-v1/weighted_alpaca_eval_gpt4_turbo/weighted_alpaca_eval_gpt4_turbo
INFO:root:Evaluating the blendaxai-gm-l6-vo31 outputs.
INFO:root:Saving all results to results/blendaxai-gm-l6-vo31/weighted_alpaca_eval_gpt4_turbo/weighted_alpaca_eval_gpt4_turbo
INFO:root:Evaluating the gemma-2-9b-it-WPO-HB outputs.
INFO:root:Saving all results to results/gemma-2-9b-it-WPO-HB/weighted_alpaca_eval_gpt4_turbo/weighted_alpaca_eval_gpt4_turbo
INFO:root:Evaluating the SelfMoA_gemma-2-9b-it-SimPO outputs.
INFO:root:Saving all results to results/SelfMoA_gemma-2-9b-it-SimPO/weighted_alpaca_eval_gpt4_turbo/weighted_alpaca_eval_gpt4_turbo
INFO:root:Evaluating the blendaxai-gm-l

Saved recomputed leaderboard → src/alpaca_eval/leaderboards/data_AlpacaEval_2/sensitivity_analysis/position_0.csv
Recomputed 218 models.
SPEARMAN CORRELATION WITH LAMBDA: 0.195
Spearman (LC AlpacaEval vs Chatbot Arena): 0.9736


INFO:root:Gathered 213 of 223 models
INFO:root:Evaluating the SelfMoA_gemma-2-9b-it-WPO-HB outputs.
INFO:root:Saving all results to results/SelfMoA_gemma-2-9b-it-WPO-HB/weighted_alpaca_eval_gpt4_turbo/weighted_alpaca_eval_gpt4_turbo
INFO:root:Evaluating the Shopee-SlimMoA-v1 outputs.
INFO:root:Saving all results to results/Shopee-SlimMoA-v1/weighted_alpaca_eval_gpt4_turbo/weighted_alpaca_eval_gpt4_turbo
INFO:root:Evaluating the blendaxai-gm-l6-vo31 outputs.
INFO:root:Saving all results to results/blendaxai-gm-l6-vo31/weighted_alpaca_eval_gpt4_turbo/weighted_alpaca_eval_gpt4_turbo
INFO:root:Evaluating the gemma-2-9b-it-WPO-HB outputs.
INFO:root:Saving all results to results/gemma-2-9b-it-WPO-HB/weighted_alpaca_eval_gpt4_turbo/weighted_alpaca_eval_gpt4_turbo
INFO:root:Evaluating the SelfMoA_gemma-2-9b-it-SimPO outputs.
INFO:root:Saving all results to results/SelfMoA_gemma-2-9b-it-SimPO/weighted_alpaca_eval_gpt4_turbo/weighted_alpaca_eval_gpt4_turbo
INFO:root:Evaluating the blendaxai-gm-l

Saved recomputed leaderboard → src/alpaca_eval/leaderboards/data_AlpacaEval_2/sensitivity_analysis/position_1.csv
Recomputed 218 models.
SPEARMAN CORRELATION WITH LAMBDA: 0.205
Spearman (LC AlpacaEval vs Chatbot Arena): 0.9752


INFO:root:Gathered 213 of 223 models
INFO:root:Evaluating the SelfMoA_gemma-2-9b-it-WPO-HB outputs.
INFO:root:Saving all results to results/SelfMoA_gemma-2-9b-it-WPO-HB/weighted_alpaca_eval_gpt4_turbo/weighted_alpaca_eval_gpt4_turbo
INFO:root:Evaluating the Shopee-SlimMoA-v1 outputs.
INFO:root:Saving all results to results/Shopee-SlimMoA-v1/weighted_alpaca_eval_gpt4_turbo/weighted_alpaca_eval_gpt4_turbo
INFO:root:Evaluating the blendaxai-gm-l6-vo31 outputs.
INFO:root:Saving all results to results/blendaxai-gm-l6-vo31/weighted_alpaca_eval_gpt4_turbo/weighted_alpaca_eval_gpt4_turbo
INFO:root:Evaluating the gemma-2-9b-it-WPO-HB outputs.
INFO:root:Saving all results to results/gemma-2-9b-it-WPO-HB/weighted_alpaca_eval_gpt4_turbo/weighted_alpaca_eval_gpt4_turbo
INFO:root:Evaluating the SelfMoA_gemma-2-9b-it-SimPO outputs.
INFO:root:Saving all results to results/SelfMoA_gemma-2-9b-it-SimPO/weighted_alpaca_eval_gpt4_turbo/weighted_alpaca_eval_gpt4_turbo
INFO:root:Evaluating the blendaxai-gm-l

Saved recomputed leaderboard → src/alpaca_eval/leaderboards/data_AlpacaEval_2/sensitivity_analysis/position_2.csv
Recomputed 218 models.
SPEARMAN CORRELATION WITH LAMBDA: 0.21
Spearman (LC AlpacaEval vs Chatbot Arena): 0.9752


INFO:root:Gathered 213 of 223 models
INFO:root:Evaluating the SelfMoA_gemma-2-9b-it-WPO-HB outputs.
INFO:root:Saving all results to results/SelfMoA_gemma-2-9b-it-WPO-HB/weighted_alpaca_eval_gpt4_turbo/weighted_alpaca_eval_gpt4_turbo
INFO:root:Evaluating the Shopee-SlimMoA-v1 outputs.
INFO:root:Saving all results to results/Shopee-SlimMoA-v1/weighted_alpaca_eval_gpt4_turbo/weighted_alpaca_eval_gpt4_turbo
INFO:root:Evaluating the blendaxai-gm-l6-vo31 outputs.
INFO:root:Saving all results to results/blendaxai-gm-l6-vo31/weighted_alpaca_eval_gpt4_turbo/weighted_alpaca_eval_gpt4_turbo
INFO:root:Evaluating the gemma-2-9b-it-WPO-HB outputs.
INFO:root:Saving all results to results/gemma-2-9b-it-WPO-HB/weighted_alpaca_eval_gpt4_turbo/weighted_alpaca_eval_gpt4_turbo
INFO:root:Evaluating the SelfMoA_gemma-2-9b-it-SimPO outputs.
INFO:root:Saving all results to results/SelfMoA_gemma-2-9b-it-SimPO/weighted_alpaca_eval_gpt4_turbo/weighted_alpaca_eval_gpt4_turbo
INFO:root:Evaluating the blendaxai-gm-l

Saved recomputed leaderboard → src/alpaca_eval/leaderboards/data_AlpacaEval_2/sensitivity_analysis/position_3.csv
Recomputed 218 models.
SPEARMAN CORRELATION WITH LAMBDA: 0.215
Spearman (LC AlpacaEval vs Chatbot Arena): 0.9752


Lamda - correlation:
1. **0.195** -- 0.9736 
2. **0.205** -- 0.9752
3. **0.210** -- 0.9752
4. **0.215** -- 0.9752

## Position component with interaction 

In [51]:
process_sensitivity_analysis(
    "np.tanh(std_delta_len) + instruction_difficulty + position_component + np.tanh(std_delta_len) * position_component + not_gamed_baseline.astype(float) - 1")

INFO:root:Gathered 213 of 223 models
INFO:root:Evaluating the SelfMoA_gemma-2-9b-it-WPO-HB outputs.
INFO:root:Saving all results to results/SelfMoA_gemma-2-9b-it-WPO-HB/weighted_alpaca_eval_gpt4_turbo/weighted_alpaca_eval_gpt4_turbo
INFO:root:Evaluating the Shopee-SlimMoA-v1 outputs.
INFO:root:Saving all results to results/Shopee-SlimMoA-v1/weighted_alpaca_eval_gpt4_turbo/weighted_alpaca_eval_gpt4_turbo
INFO:root:Evaluating the blendaxai-gm-l6-vo31 outputs.
INFO:root:Saving all results to results/blendaxai-gm-l6-vo31/weighted_alpaca_eval_gpt4_turbo/weighted_alpaca_eval_gpt4_turbo
INFO:root:Evaluating the gemma-2-9b-it-WPO-HB outputs.
INFO:root:Saving all results to results/gemma-2-9b-it-WPO-HB/weighted_alpaca_eval_gpt4_turbo/weighted_alpaca_eval_gpt4_turbo
INFO:root:Evaluating the SelfMoA_gemma-2-9b-it-SimPO outputs.
INFO:root:Saving all results to results/SelfMoA_gemma-2-9b-it-SimPO/weighted_alpaca_eval_gpt4_turbo/weighted_alpaca_eval_gpt4_turbo
INFO:root:Evaluating the blendaxai-gm-l

Saved recomputed leaderboard → src/alpaca_eval/leaderboards/data_AlpacaEval_2/sensitivity_analysis/position_0.csv
Recomputed 218 models.


SPEARMAN CORRELATION WITH LAMBDA: 0.195
Spearman (LC AlpacaEval vs Chatbot Arena): 0.9739


INFO:root:Gathered 213 of 223 models
INFO:root:Evaluating the SelfMoA_gemma-2-9b-it-WPO-HB outputs.
INFO:root:Saving all results to results/SelfMoA_gemma-2-9b-it-WPO-HB/weighted_alpaca_eval_gpt4_turbo/weighted_alpaca_eval_gpt4_turbo
INFO:root:Evaluating the Shopee-SlimMoA-v1 outputs.
INFO:root:Saving all results to results/Shopee-SlimMoA-v1/weighted_alpaca_eval_gpt4_turbo/weighted_alpaca_eval_gpt4_turbo
INFO:root:Evaluating the blendaxai-gm-l6-vo31 outputs.
INFO:root:Saving all results to results/blendaxai-gm-l6-vo31/weighted_alpaca_eval_gpt4_turbo/weighted_alpaca_eval_gpt4_turbo
INFO:root:Evaluating the gemma-2-9b-it-WPO-HB outputs.
INFO:root:Saving all results to results/gemma-2-9b-it-WPO-HB/weighted_alpaca_eval_gpt4_turbo/weighted_alpaca_eval_gpt4_turbo
INFO:root:Evaluating the SelfMoA_gemma-2-9b-it-SimPO outputs.
INFO:root:Saving all results to results/SelfMoA_gemma-2-9b-it-SimPO/weighted_alpaca_eval_gpt4_turbo/weighted_alpaca_eval_gpt4_turbo
INFO:root:Evaluating the blendaxai-gm-l

Saved recomputed leaderboard → src/alpaca_eval/leaderboards/data_AlpacaEval_2/sensitivity_analysis/position_1.csv
Recomputed 218 models.
SPEARMAN CORRELATION WITH LAMBDA: 0.205
Spearman (LC AlpacaEval vs Chatbot Arena): 0.9734


INFO:root:Gathered 213 of 223 models
INFO:root:Evaluating the SelfMoA_gemma-2-9b-it-WPO-HB outputs.
INFO:root:Saving all results to results/SelfMoA_gemma-2-9b-it-WPO-HB/weighted_alpaca_eval_gpt4_turbo/weighted_alpaca_eval_gpt4_turbo
INFO:root:Evaluating the Shopee-SlimMoA-v1 outputs.
INFO:root:Saving all results to results/Shopee-SlimMoA-v1/weighted_alpaca_eval_gpt4_turbo/weighted_alpaca_eval_gpt4_turbo
INFO:root:Evaluating the blendaxai-gm-l6-vo31 outputs.
INFO:root:Saving all results to results/blendaxai-gm-l6-vo31/weighted_alpaca_eval_gpt4_turbo/weighted_alpaca_eval_gpt4_turbo
INFO:root:Evaluating the gemma-2-9b-it-WPO-HB outputs.
INFO:root:Saving all results to results/gemma-2-9b-it-WPO-HB/weighted_alpaca_eval_gpt4_turbo/weighted_alpaca_eval_gpt4_turbo
INFO:root:Evaluating the SelfMoA_gemma-2-9b-it-SimPO outputs.
INFO:root:Saving all results to results/SelfMoA_gemma-2-9b-it-SimPO/weighted_alpaca_eval_gpt4_turbo/weighted_alpaca_eval_gpt4_turbo
INFO:root:Evaluating the blendaxai-gm-l

Saved recomputed leaderboard → src/alpaca_eval/leaderboards/data_AlpacaEval_2/sensitivity_analysis/position_2.csv
Recomputed 218 models.
SPEARMAN CORRELATION WITH LAMBDA: 0.21
Spearman (LC AlpacaEval vs Chatbot Arena): 0.9734


INFO:root:Gathered 213 of 223 models
INFO:root:Evaluating the SelfMoA_gemma-2-9b-it-WPO-HB outputs.
INFO:root:Saving all results to results/SelfMoA_gemma-2-9b-it-WPO-HB/weighted_alpaca_eval_gpt4_turbo/weighted_alpaca_eval_gpt4_turbo
INFO:root:Evaluating the Shopee-SlimMoA-v1 outputs.
INFO:root:Saving all results to results/Shopee-SlimMoA-v1/weighted_alpaca_eval_gpt4_turbo/weighted_alpaca_eval_gpt4_turbo
INFO:root:Evaluating the blendaxai-gm-l6-vo31 outputs.
INFO:root:Saving all results to results/blendaxai-gm-l6-vo31/weighted_alpaca_eval_gpt4_turbo/weighted_alpaca_eval_gpt4_turbo
INFO:root:Evaluating the gemma-2-9b-it-WPO-HB outputs.
INFO:root:Saving all results to results/gemma-2-9b-it-WPO-HB/weighted_alpaca_eval_gpt4_turbo/weighted_alpaca_eval_gpt4_turbo
INFO:root:Evaluating the SelfMoA_gemma-2-9b-it-SimPO outputs.
INFO:root:Saving all results to results/SelfMoA_gemma-2-9b-it-SimPO/weighted_alpaca_eval_gpt4_turbo/weighted_alpaca_eval_gpt4_turbo
INFO:root:Evaluating the blendaxai-gm-l

Saved recomputed leaderboard → src/alpaca_eval/leaderboards/data_AlpacaEval_2/sensitivity_analysis/position_3.csv
Recomputed 218 models.
SPEARMAN CORRELATION WITH LAMBDA: 0.215
Spearman (LC AlpacaEval vs Chatbot Arena): 0.9734


Lamda - correlation:
1. **0.195** -- 0.9739
2. **0.205** -- 0.9734
3. **0.210** -- 0.9734
4. **0.215** -- 0.9734

# DoubleML position implementation

In [9]:
model_output_list = get_all_res(LC_SRC_LB)
len(model_output_list)

INFO:root:Gathered 213 of 223 models


213

In [10]:
all_model_outputs = pd.concat([pd.read_json(output_path) for output_path in model_output_list])

all_model_outputs

,instruction,dataset,output,generator,datasplit,original_output,id,metadata
0,What are the names of some famous actors that ...,helpful_base,Here's a definitive list of A-list actors whos...,SelfMoA_gemma-2-9b-it-WPO-HB,NaN,NaN,NaN,NaN
1,How did US states get their names?,helpful_base,Here's a comprehensive look at the fascinating...,SelfMoA_gemma-2-9b-it-WPO-HB,NaN,NaN,NaN,NaN
2,"Hi, my sister and her girlfriends want me to p...",helpful_base,Here's a **Kickball Crash Course** designed t...,SelfMoA_gemma-2-9b-it-WPO-HB,NaN,NaN,NaN,NaN
3,What is some cool music from the 1920s?,helpful_base,"Here's a curated list of ""cool"" music from t...",SelfMoA_gemma-2-9b-it-WPO-HB,NaN,NaN,NaN,NaN
4,How do I wrap a present neatly?,helpful_base,"Here's a refined, step-by-step guide to wrappi...",SelfMoA_gemma-2-9b-it-WPO-HB,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...
800,Write a script for a YouTube video exploring t...,vicuna,"[Introduction]\n\nHello, welcome to our journe...",baichuan-13b-chat,NaN,NaN,NaN,NaN
801,Compose an engaging travel blog post about a r...,vicuna,Title: A Journey of Cultural Exploration in Pa...,baichuan-13b-chat,NaN,NaN,NaN,NaN
802,Write a captivating movie review for a recentl...,vicuna,"Title: ""The Last Starfighter""\n\nGenre: Scienc...",baichuan-13b-chat,NaN,NaN,NaN,NaN
803,Structure a podcast script for an episode disc...,vicuna,Intro (0:00-0:20)\n\n[Theme song plays]\n\nHos...,baichuan-13b-chat,NaN,NaN,NaN,NaN


In [11]:
all_annotations = pd.concat([pd.read_json(Path(output_path).parent / "weighted_alpaca_eval_gpt4_turbo" / "annotations.json") for output_path in model_output_list])

all_annotations

,instruction,output_1,generator_1,dataset,output_2,generator_2,annotator,preference,preference_version,preference_price_per_example,...,preference_date,time_per_example,raw_completion,price_per_example,datasplit,input,sample_mode,original_output,id,metadata
0,What are the names of some famous actors that ...,Several famous actors started their careers on...,gpt4_1106_preview,helpful_base,Here's a definitive list of A-list actors whos...,SelfMoA_gemma-2-9b-it-WPO-HB,weighted_alpaca_eval_gpt4_turbo,1.989999,alpaca_eval==0.6.5,0.01414,...,2024-09-23T22:38:57.204834,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,How did US states get their names?,The names of U.S. states are derived from a va...,gpt4_1106_preview,helpful_base,Here's a comprehensive look at the fascinating...,SelfMoA_gemma-2-9b-it-WPO-HB,weighted_alpaca_eval_gpt4_turbo,1.754209,alpaca_eval==0.6.5,0.01687,...,2024-09-23T22:38:57.204834,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,"Hi, my sister and her girlfriends want me to p...",Kickball is a fun and simple game that is simi...,gpt4_1106_preview,helpful_base,Here's a **Kickball Crash Course** designed t...,SelfMoA_gemma-2-9b-it-WPO-HB,weighted_alpaca_eval_gpt4_turbo,1.727014,alpaca_eval==0.6.5,0.01687,...,2024-09-23T22:38:57.204834,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,What is some cool music from the 1920s?,"The 1920s, often referred to as the ""Roaring T...",gpt4_1106_preview,helpful_base,"Here's a curated list of ""cool"" music from t...",SelfMoA_gemma-2-9b-it-WPO-HB,weighted_alpaca_eval_gpt4_turbo,1.765762,alpaca_eval==0.6.5,0.01620,...,2024-09-23T22:38:57.204834,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,How do I wrap a present neatly?,Wrapping a present neatly can be quite straigh...,gpt4_1106_preview,helpful_base,"Here's a refined, step-by-step guide to wrappi...",SelfMoA_gemma-2-9b-it-WPO-HB,weighted_alpaca_eval_gpt4_turbo,1.999200,alpaca_eval==0.6.5,0.01461,...,2024-09-23T22:38:57.204834,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
800,Write a script for a YouTube video exploring t...,[Intro]\n(Upbeat jazz music playing softly in ...,gpt4_1106_preview,vicuna,"[Introduction]\n\nHello, welcome to our journe...",baichuan-13b-chat,weighted_alpaca_eval_gpt4_turbo,1.000005,NaN,NaN,...,NaN,0.189929,"{'finish_reason': 'length', 'index': 0, 'logpr...",0.01933,NaN,NaN,NaN,NaN,NaN,NaN
801,Compose an engaging travel blog post about a r...,Title: Aloha Spirit: A Journey Through the Hea...,gpt4_1106_preview,vicuna,Title: A Journey of Cultural Exploration in Pa...,baichuan-13b-chat,weighted_alpaca_eval_gpt4_turbo,1.000001,NaN,NaN,...,NaN,0.189929,"{'finish_reason': 'length', 'index': 0, 'logpr...",0.02138,NaN,NaN,NaN,NaN,NaN,NaN
802,Write a captivating movie review for a recentl...,"Title: ""Eclipse of Tomorrow""\n\nIn the pantheo...",gpt4_1106_preview,vicuna,"Title: ""The Last Starfighter""\n\nGenre: Scienc...",baichuan-13b-chat,weighted_alpaca_eval_gpt4_turbo,1.000064,NaN,NaN,...,NaN,0.189929,"{'finish_reason': 'length', 'index': 0, 'logpr...",0.01185,NaN,NaN,NaN,NaN,NaN,NaN
803,Structure a podcast script for an episode disc...,"# Podcast Episode Script: ""The Streaming Revol...",gpt4_1106_preview,vicuna,Intro (0:00-0:20)\n\n[Theme song plays]\n\nHos...,baichuan-13b-chat,weighted_alpaca_eval_gpt4_turbo,1.000003,NaN,NaN,...,NaN,0.189929,"{'finish_reason': 'length', 'index': 0, 'logpr...",0.01538,NaN,NaN,NaN,NaN,NaN,NaN


In [12]:
all_model_outputs.to_json("results/all/model_outputs.json", orient="records")
all_annotations.to_json("results/all/weighted_alpaca_eval_gpt4_turbo/annotations.json", orient="records")

In [14]:
rows_lb = evaluate_multiple(
    model_outputs=all_model_outputs,
    name="all",
    reference_outputs=constants.ALPACAEVAL_REFERENCE_OUTPUTS,
    annotators_config=constants.DEFAULT_ANNOTATOR_CONFIG,
    fn_metric="get_doubleml_length_position_controlled_winrate",
    sort_by="length_controlled_winrate",
    output_path="auto",
    is_recompute_metrics_only=True,
)

INFO:root:Evaluating multiple models from dataset: all
INFO:root:Loaded 171453 model outputs and 805 reference outputs
INFO:root:Found 213 models: ['SelfMoA_gemma-2-9b-it-WPO-HB' 'Shopee-SlimMoA-v1' 'blendaxai-gm-l6-vo31'
 'gemma-2-9b-it-WPO-HB' 'SelfMoA_gemma-2-9b-it-SimPO'
 'blendaxai-gm-l3-v35' 'gemma-2-9b-it-SimPO' 'TOA'
 'FuseChat-Gemma-2-9B-Instruct' 'openpipe-moa-gpt-4-turbo-v1'
 'gemma-2-9b-it-DPO' 'FuseChat-Llama-3.1-8B-Instruct' 'Together-MoA'
 'FuseChat-Qwen-2.5-7B-Instruct' 'Llama3-PBM-Nova-70B'
 'Storm-7B-best-of-64' 'Together-MoA-Lite' 'gpt-4o-2024-05-13'
 'higgs-llama-3-70b-v2' 'FuseChat-Llama-3.2-3B-Instruct'
 'SPPO-Gemma-2-9B-It-PairRM' 'Llama-3-Instruct-8B-WPO-HB-v2'
 'claude-3-5-sonnet-20240620' 'gpt4_1106_preview_verbose'
 'gpt-4o-mini-2024-07-18' 'Storm-7B' 'gpt4_1106_preview'
 'REBEL-Llama-3-8B-Instruct-Armo' 'Infinity-Instruct-7M-Gen-Llama3_1-70B'
 'Llama-3-Instruct-8B-SimPO-ExPO' 'Llama-3-Instruct-8B-SimPO'
 'Nanbeige-Plus-Chat-v0.1' 'Qwen1.5-110B-Chat'
 'Llama-

[CV] END learning_rate=0.05, max_depth=3, min_samples_leaf=5, min_samples_split=10, n_estimators=100, subsample=0.8; total time=   0.1s
[CV] END learning_rate=0.05, max_depth=3, min_samples_leaf=5, min_samples_split=10, n_estimators=100, subsample=1.0; total time=   0.1s
[CV] END learning_rate=0.05, max_depth=3, min_samples_leaf=5, min_samples_split=10, n_estimators=150, subsample=0.9; total time=   0.1s
[CV] END learning_rate=0.05, max_depth=3, min_samples_leaf=5, min_samples_split=10, n_estimators=200, subsample=0.8; total time=   0.1s
[CV] END learning_rate=0.05, max_depth=3, min_samples_leaf=5, min_samples_split=10, n_estimators=200, subsample=1.0; total time=   0.1s
[CV] END learning_rate=0.05, max_depth=3, min_samples_leaf=5, min_samples_split=20, n_estimators=100, subsample=1.0; total time=   0.1s
[CV] END learning_rate=0.05, max_depth=3, min_samples_leaf=5, min_samples_split=20, n_estimators=150, subsample=0.9; total time=   0.1s
[CV] END learning_rate=0.05, max_depth=3, min_sa

INFO:root:[sklearn] Fitting 5 folds for each of 324 candidates, totalling 1620 fits
INFO:root:[sklearn] Fitting 5 folds for each of 324 candidates, totalling 1620 fits
INFO:root:[sklearn] Fitting 5 folds for each of 324 candidates, totalling 1620 fits


[CV] END learning_rate=0.15, max_depth=4, min_samples_leaf=10, min_samples_split=20, n_estimators=100, subsample=1.0; total time=   0.1s
[CV] END learning_rate=0.15, max_depth=4, min_samples_leaf=10, min_samples_split=20, n_estimators=100, subsample=1.0; total time=   0.1s
[CV] END learning_rate=0.15, max_depth=4, min_samples_leaf=10, min_samples_split=20, n_estimators=200, subsample=0.8; total time=   0.1s
[CV] END learning_rate=0.15, max_depth=4, min_samples_leaf=10, min_samples_split=20, n_estimators=200, subsample=0.8; total time=   0.2s
[CV] END learning_rate=0.15, max_depth=5, min_samples_leaf=5, min_samples_split=10, n_estimators=100, subsample=0.9; total time=   0.1s
[CV] END learning_rate=0.15, max_depth=5, min_samples_leaf=5, min_samples_split=10, n_estimators=100, subsample=0.9; total time=   0.1s
[CV] END learning_rate=0.15, max_depth=5, min_samples_leaf=5, min_samples_split=10, n_estimators=150, subsample=1.0; total time=   0.1s
[CV] END learning_rate=0.15, max_depth=5, mi

INFO:root:[sklearn] Fitting 5 folds for each of 324 candidates, totalling 1620 fits
INFO:root:[sklearn] Fitting 5 folds for each of 324 candidates, totalling 1620 fits
INFO:root:[sklearn] Fitting 5 folds for each of 324 candidates, totalling 1620 fits


[CV] END learning_rate=0.15, max_depth=4, min_samples_leaf=5, min_samples_split=20, n_estimators=200, subsample=1.0; total time=  26.7s
[CV] END learning_rate=0.15, max_depth=4, min_samples_leaf=10, min_samples_split=10, n_estimators=100, subsample=0.9; total time=  12.2s
[CV] END learning_rate=0.15, max_depth=4, min_samples_leaf=10, min_samples_split=10, n_estimators=150, subsample=0.8; total time=  16.6s
[CV] END learning_rate=0.15, max_depth=4, min_samples_leaf=10, min_samples_split=10, n_estimators=150, subsample=1.0; total time=  20.0s
[CV] END learning_rate=0.15, max_depth=4, min_samples_leaf=10, min_samples_split=10, n_estimators=200, subsample=0.9; total time=  24.9s
[CV] END learning_rate=0.15, max_depth=4, min_samples_leaf=10, min_samples_split=20, n_estimators=100, subsample=0.8; total time=  11.0s
[CV] END learning_rate=0.15, max_depth=4, min_samples_leaf=10, min_samples_split=20, n_estimators=100, subsample=1.0; total time=  13.3s
[CV] END learning_rate=0.15, max_depth=4, 

ERROR:root:DoubleML model training failed.
Traceback (most recent call last):
  File "/Users/simon/PycharmProjects/alpaca_eval/src/alpaca_eval/metrics/doubleml_winrate.py", line 145, in get_doubleml_length_position_controlled_winrate
    dml = _setup_and_fit_dml_model(featured_df, ml_g, ml_m, n_folds, n_rep)
  File "/Users/simon/PycharmProjects/alpaca_eval/src/alpaca_eval/metrics/doubleml_winrate.py", line 342, in _setup_and_fit_dml_model
    dml.fit(store_predictions=True)
  File "/Users/simon/PycharmProjects/alpaca_eval/.venv/lib/python3.9/site-packages/doubleml/irm/apos.py", line 394, in fit
    fitted_models = parallel(
  File "/Users/simon/PycharmProjects/alpaca_eval/.venv/lib/python3.9/site-packages/joblib/parallel.py", line 1986, in __call__
    return output if self.return_generator else list(output)
  File "/Users/simon/PycharmProjects/alpaca_eval/.venv/lib/python3.9/site-packages/joblib/parallel.py", line 1914, in _get_sequential_output
    res = func(*args, **kwargs)
  File 

TypeError: log_loss() got an unexpected keyword argument 'needs_proba'

In [40]:
lb_doubleml_path_pos = "src/alpaca_eval/leaderboards/recomputed_weighted_alpaca_eval_gpt4_turbo_positioned_doubleml.csv"

rows_lb = rows_lb.set_index('model')

rows_lb.to_csv(lb_doubleml_path_pos)
rows_lb

,win_rate,standard_error,n_wins,n_wins_base,n_draws,n_total,discrete_win_rate,length_controlled_winrate,lc_standard_error
model,,,,,,,,,
gpt4_1106_preview,50.000000,0.000000,0,0,805,805,50.000000,50.000000,0.000000
Shopee-SlimMoA-v1,75.614287,1.270627,621,184,0,805,77.142857,30.997289,1.200953
gemma-2-9b-it-WPO-HB,77.825032,1.235586,640,163,2,805,79.627329,30.853349,1.166450
blendaxai-gm-l3-v35,73.410357,1.254951,607,196,2,805,75.527950,27.527374,1.052815
blendaxai-gm-l6-vo31,69.110335,1.328074,562,242,1,805,69.875776,27.525829,1.124397
...,...,...,...,...,...,...,...,...,...
ultralm-13b-best-of-16,11.307315,0.941843,80,723,2,805,10.062112,-8.730537,0.946422
guanaco-7b,2.880002,0.520292,21,783,1,805,2.670807,-8.763964,0.600664
baichuan-13b-chat,1.992146,0.417699,14,790,1,805,1.801242,-9.167245,0.530543


In [41]:
calculate_spearman_correlation(lb_doubleml_path_pos)

Spearman (LC AlpacaEval vs Chatbot Arena): 0.9274
